# N-body simulation on the GPU

Direct O(N²) gravity (naive kernel and shared-memory tiled kernel) and a Barnes-Hut version, timed for increasing N.

Runtime → Change runtime type → **T4 GPU**, then Run all. The first cell asks for `nbody_cuda.zip`.

## 1. Setup and build

In [ ]:
import os, subprocess
os.chdir('/content')

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nvcc --version | tail -2

if not os.path.isdir('nbody_cuda'):
    from google.colab import files
    up = files.upload()                # choose nbody_cuda.zip
    zipname = list(up)[0]
    !unzip -q -o "{zipname}"
%cd /content/nbody_cuda

# compile for the GPU we actually got (T4 -> sm_75)
cc = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
                    capture_output=True, text=True).stdout.strip().split('\n')[0].replace('.', '')
arch = f'-arch=sm_{cc}' if cc else '-arch=native'
!make clean && make ARCH={arch}

## 2. Correctness check

Small run (N = 2000, 50 steps) with every version, compared against the double-precision CPU code.
N = 2000 is not a multiple of the block size on purpose, so the partial last tile gets tested too.
Barnes-Hut with θ = 0 opens every cell, so it should give the exact direct-sum answer.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs('inputs', exist_ok=True)
os.makedirs('results', exist_ok=True)

def make_input(n, steps, dt=1e-3):
    path = f'inputs/n{n}_s{steps}.txt'
    if not os.path.exists(path):
        subprocess.run(['python3', 'gen_input.py', str(n), path, '--steps', str(steps),
                        '--dt', str(dt)], check=True)
    return path

def run(cmd, reps=1):
    # runs the program `reps` times and returns the [summary] line of the fastest run
    best = None
    for _ in range(reps):
        p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if p.returncode != 0:
            print(p.stdout, p.stderr)
            raise RuntimeError('failed: ' + cmd)
        line = [l for l in p.stdout.splitlines() if l.startswith('[summary]')][-1]
        r = {}
        for kv in line.split()[1:]:
            k, v = kv.split('=')
            for conv in (int, float):
                try:
                    v = conv(v)
                    break
                except ValueError:
                    pass
            r[k] = v
        if best is None or r['time_ms'] < best['time_ms']:
            best = r
    return best

def load_state(path):
    rows = [l for l in open(path) if l.strip() and not l.startswith('#')]
    return np.loadtxt(rows[1:])        # rows[0] is the "N steps dt G eps" line

inp = make_input(2000, 50)
checks = [('cpu (double)',      f'./nbody {inp} results/check_cpu.txt cpu'),
          ('gpu naive',         f'./nbody {inp} results/check_naive.txt naive 256'),
          ('gpu tiled',         f'./nbody {inp} results/check_tiled.txt tiled 256'),
          ('barnes-hut θ=0',    f'./barnes_hut {inp} results/check_bh0.txt 0.0'),
          ('barnes-hut θ=0.5',  f'./barnes_hut {inp} results/check_bh05.txt 0.5')]

rows = []
for name, cmd in checks:
    r = run(cmd)
    s = load_state(cmd.split()[2])
    if not rows:
        ref = s
    rows.append({'version': name,
                 'max |Δr| vs cpu': np.abs(s[:, :3] - ref[:, :3]).max(),
                 'max |Δv| vs cpu': np.abs(s[:, 3:6] - ref[:, 3:6]).max(),
                 'time (ms)': r['time_ms']})

moved = np.linalg.norm(ref[:, :3] - load_state(inp)[:, :3], axis=1).mean()
print(f'particles moved {moved:.2e} on average during the run (for scale)')
pd.DataFrame(rows)

## 3. Execution time vs N

10 time steps for each N, best of 3 runs. The CPU version is only run up to N = 16384 because it gets too slow.
`ms_per_step` is the time of the time-step loop divided by the number of steps.

In [ ]:
Ns = [1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072, 262144]
STEPS = 10

rec = []
for n in Ns:
    inp = make_input(n, STEPS)
    rec.append(run(f'./nbody {inp} results/tmp.txt naive 256', reps=3))
    rec.append(run(f'./nbody {inp} results/tmp.txt tiled 256', reps=3))
    rec.append(run(f'./barnes_hut {inp} results/tmp.txt 0.5', reps=3))
    if n <= 16384:
        rec.append(run(f'./nbody {inp} results/tmp.txt cpu'))
    print('done N =', n)

scal = pd.DataFrame(rec)
scal.to_csv('results/scaling.csv', index=False)

t = scal.pivot(index='n', columns='mode', values='ms_per_step')[['cpu', 'naive', 'tiled', 'bh']]
t['tiled vs naive'] = t['naive'] / t['tiled']
t['tiled vs cpu'] = t['cpu'] / t['tiled']
t['bh vs tiled'] = t['tiled'] / t['bh']
t.round(3)

## 4. Block size, and Barnes-Hut θ

(a) Tile/block size for the tiled kernel at N = 65536.
(b) Barnes-Hut for different θ at N = 32768 over 200 steps. Accuracy is measured two ways: the relative force
error against the exact sum at the first step (`err_mean`, `err_max`), and how far the final positions are from
the tiled direct-sum run (`pos_err`, mean distance, the cluster has radius 1).

In [ ]:
inp = make_input(65536, STEPS)
blk = pd.DataFrame([run(f'./nbody {inp} results/tmp.txt tiled {b}', reps=3)
                    for b in [32, 64, 128, 256, 512, 1024]])
blk.to_csv('results/block_size.csv', index=False)
print(blk[['block', 'ms_per_step', 'gflops']].to_string(index=False), '\n')

inp = make_input(32768, 200)
direct = run(f'./nbody {inp} results/ref_tiled.txt tiled 256')
ref = load_state('results/ref_tiled.txt')
print(f"direct (tiled) run: {direct['ms_per_step']:.3f} ms per step")

th = []
for theta in [0.2, 0.3, 0.4, 0.5, 0.7, 0.9, 1.2]:
    r = run(f'./barnes_hut {inp} results/tmp_bh.txt {theta}')
    s = load_state('results/tmp_bh.txt')
    r['pos_err'] = np.linalg.norm(s[:, :3] - ref[:, :3], axis=1).mean()
    r['speedup'] = direct['time_ms'] / r['time_ms']
    th.append(r)
theta_df = pd.DataFrame(th)
theta_df.to_csv('results/theta.csv', index=False)
theta_df[['theta', 'ms_per_step', 'build_ms', 'kernel_ms', 'avg_inter',
          'err_mean', 'err_max', 'pos_err', 'speedup']]

## 5. Graphs

In [ ]:
names = {'cpu': 'CPU, 1 core (double)', 'naive': 'GPU naive', 'tiled': 'GPU tiled',
         'bh': 'GPU Barnes-Hut, θ = 0.5'}
n_ = np.array(Ns, dtype=float)

fig, ax = plt.subplots(2, 2, figsize=(12, 9))

a = ax[0, 0]
for m in ['cpu', 'naive', 'tiled', 'bh']:
    s = t[m].dropna()
    a.loglog(s.index, s.values, 'o-', label=names[m])
a.loglog(n_, t['tiled'].iloc[-1] * (n_ / n_[-1]) ** 2, 'k--', lw=1, label='∝ N²')
a.loglog(n_, t['bh'].iloc[-1] * (n_ * np.log2(n_)) / (n_[-1] * np.log2(n_[-1])), 'k:', lw=1,
         label='∝ N log N')
a.set(xlabel='N', ylabel='time per step (ms)', title='Execution time vs N')
a.grid(True, which='both', alpha=0.3)
a.legend(fontsize=8)

a = ax[0, 1]
a.loglog(t.index, t['tiled vs cpu'], 'o-', label='tiled vs CPU')
a.loglog(t.index, t['tiled vs naive'], 's-', label='tiled vs naive')
a.loglog(t.index, t['bh vs tiled'], '^-', label='Barnes-Hut vs tiled')
a.axhline(1, color='gray', lw=1)
a.set(xlabel='N', ylabel='speedup (×)', title='Speedups')
a.grid(True, which='both', alpha=0.3)
a.legend(fontsize=8)

g = scal.pivot(index='n', columns='mode', values='gflops')
a = ax[1, 0]
a.semilogx(g.index, g['naive'], 'o-', label='naive')
a.semilogx(g.index, g['tiled'], 's-', label='tiled')
a.set(xlabel='N', ylabel='GFLOP/s (20 flops per interaction)', title='Throughput of the O(N²) kernels')
a.grid(alpha=0.3)
a.legend()

a = ax[1, 1]
a.plot(theta_df['theta'], theta_df['ms_per_step'], 'o-', color='C0')
a.set_xlabel('θ')
a.set_ylabel('time per step (ms)', color='C0')
b = a.twinx()
b.semilogy(theta_df['theta'], theta_df['err_mean'], 's--', color='C3')
b.set_ylabel('mean relative force error', color='C3')
a.set_title('Barnes-Hut speed vs accuracy (N = 32768)')
a.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/fig_scaling.png', dpi=150)
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar([str(b) for b in blk['block']], blk['gflops'])
ax[0].set(xlabel='block size (= tile size)', ylabel='GFLOP/s', title='Tiled kernel, N = 65536')

bh = scal[scal['mode'] == 'bh'].set_index('n')
parts = pd.DataFrame({'tree build (CPU)': bh['build_ms'], 'tree walk (GPU)': bh['kernel_ms'],
                      'copies / other': bh['other_ms']})
parts = 100 * parts.div(parts.sum(axis=1), axis=0)
parts.index = [str(n) for n in parts.index]
parts.plot.bar(stacked=True, ax=ax[1], rot=45)
ax[1].set(xlabel='N', ylabel='% of time per step', title='Where the Barnes-Hut time goes')
ax[1].legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.savefig('results/fig_blocksize_bh.png', dpi=150)
plt.show()

## 6. Sample run and download the results

In [ ]:
!python3 gen_input.py 512 results/sample_input.txt --steps 100
!./nbody results/sample_input.txt results/sample_output_tiled.txt tiled 256
!./barnes_hut results/sample_input.txt results/sample_output_bh.txt 0.5
!head -4 results/sample_output_tiled.txt

!nvidia-smi --query-gpu=name --format=csv,noheader > results/gpu.txt
!rm -f results/tmp*.txt results/ref_tiled.txt
!zip -qr /content/results.zip results
from google.colab import files
files.download('/content/results.zip')